# Automatic translation to Tatar using DeepSeek

In [ ]:
# Mount Google Drive (optional — only needed if your files live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import time
import requests

In [ ]:
from google.colab import userdata

DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

In [ ]:
BASE_DIR = "<your-base-dir>"
INPUT_PATH = f"{BASE_DIR}/input/train.conll"
OUTPUT_PATH = f"{BASE_DIR}/output/train_tat.conll"
LOG_PATH = f"{BASE_DIR}/logs/errors.log"

In [ ]:
os.makedirs(f"{BASE_DIR}/input", exist_ok=True)
os.makedirs(f"{BASE_DIR}/output", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)

# create / clear the output file
open(OUTPUT_PATH, "w", encoding="utf-8").close()

In [ ]:
os.makedirs(f"{BASE_DIR}/input", exist_ok=True)
os.makedirs(f"{BASE_DIR}/output", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)

# create / clear the output file
open(OUTPUT_PATH, "w", encoding="utf-8").close()

System prompt

In [ ]:
SYSTEM_PROMPT = """СТРОГО:

Правила перевода:

– НЕ добавляй комментарии
– Выводи ТОЛЬКО CONLL формате

1. Ответ ВСЕГДА начинается со строк:
		# text: <татарский перевод>
		# intent: <intent из исходного примера>
1. Запрет есть только на связку «миңа» + «минем».
2. Время дели на сущности. Например, 9:15 9 I-datetime, : I-datetime, 15 I-datetime. 8:45кә 8 I-alarm/alarm_modifier, : I-alarm/alarm_modifier, 45к-ә I-alarm/alarm_modifier.
3. При переводе временных слотов datetime выбор формы и падежа обязан определяться семантической ролью времени.  14.1. Целевое время (установка / планирование: set / add / schedule / alarm / reminder и т. п.):используется юнәлеш килеше (-ка / -кә). При наличии am / pm / morning / evening / night запрещено использовать наречия иртән, кичен, төнлә, көндез; обязательно использовать формуиртәнге / кичке / төнге / көндезге + число + -ка / -кә. Пример: at 5 am → иртәнге 5-кә, at 7 pm → кичке 7-гә. 14.2. Описательное время (момент события, факт, описание): используется урын-вакыт килеше (-та / -тә); допускаются формы иртән, кичен, төнлә, көндез.
4. ЭТО САМОЕ ПРИОРИТЕТНОЕ ПРАВИЛО: При переводе используй максимально дословный перевод каждого токена (слова) с английского на татарский, сохраняя порядок и грамматическую форму (повелительное наклонение, время, число), если это не нарушает базовые правила татарского языка.
5. ВТОРОЕ САМОЕ ВАЖНЫЕ ПРАВИЛА:  Имена собственные (люди, артисты, авторы и так далее) ОБЯЗАТЕЛЬНО пиши кириллицей (транслитерация), с татарскими окончаниями при необходимости. Названия объектов (плейлисты, сервисы, бренды, произведения и так далее) сохраняют оригинальное написание, то есть пиши на английском.
Пример 1:
text EN: Add the Matt Murphy tune to the Flow Español playlist.
text TAT: Мэтт Мерфиның көйен Flow Español плейлистына өстә.
Пример 2:
text EN: I want to listen to something on Youtube
text TAT: Мин Youtube-та берәр нәрсә тыңларга телим
Пример3:
Matt Murphy — Мэтт Мерфиның, где «-ның» окончание из татарского языка.
8. Также можно перевести названия фильмов, сериалов и так далее, если эти произведения известны обывателю России.
9. Пример сериала: Friends — Друзья
10. Местоимение my переводи СТРОГО как притяжательный аффикс -м , -ем , -ым.
11. В изначальном английском примере «my» обозначается как reference. В татарской версии reference будем обозначать то слово, в котором есть притяжательный аффикс -м , -ем , -ым. Пример: будильнигымны сүндер, где будильнигымны B-reference, а сүндер O.
12. Сущности слота должны идти подряд как в оригинальной версии. Сущность это название, например, datetime, а слоты это каждое слово в сущности, то есть B-datetime и I-datetime означает, что в одной сущности два слота. Если слоты разделяются во время перевода на татарский, то нужно перехразировать предложение, чтобы слот был целый.
Пример:
1	бүген	B-datetime
2	Будильнигымны	O
3	кичке	B-datetime
4	5	I-datetime
5	куй	O
Перефразируем:
1	Будильнигымны	O
2	бүген	B-datetime
3	кичке	I-datetime
4	5	I-datetime
5	куй	O
13. Замени слова/названия с ЛГБТ контекстом, на нейтральные слова/названия.
14. ЧИСЛА ВСЕГДА пиши цифрами. Через дефис добавляй падежное окончание. 	НИКОГДА не заменяй числа словами (дүрт, биш, ун). Пример: 6-дан, 4-тә, 5-кә.
15. Строго ее добавляй служебные слова, если их нет оригинале.
16. Если между сущностями в исходном тексте есть логическая связь (принадлежность, авторство, назначение, источник), в татарском переводе эта связь ОБЯЗАНА быть выражена явно (через родительный падеж, притяжательный аффикс или эквивалентную конструкцию).

Пример английский

# text: Add a reminder for today at 4pm
# intent: reminder/set_reminder
1 Add O
2 a O
3 reminder O
4 for O
5 today B-datetime
6 at I-datetime
7 4pm I-datetime

Татарский перевод

# text: бүген сәгать дүрткә искәртмә өстә
# intent: reminder/set_reminder
1 бүген B-datetime
2 сәгать I-datetime
3 дүрткә I-datetime
4 искәртмә O
5 өстә O
"""

In [ ]:
def parse_conll(path):
    examples = []
    current = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if line == "":
                if current:
                    examples.append(current)
                    current = []
            else:
                current.append(line)

        if current:
            examples.append(current)

    return examples

In [ ]:
def split_id_and_content(example_lines):
    ex_id = None
    content = []

    for line in example_lines:
        if line.startswith("# id:"):
            ex_id = line
        else:
            content.append(line)

    return ex_id, "\n".join(content)

In [ ]:
def deepseek_chat(system_prompt, user_content, max_tokens=800):
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"

    payload = {
        "model": "deepseek-chat",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, json=payload, timeout=60)
    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


In [ ]:
def get_last_processed_id(output_path):
    if not os.path.exists(output_path):
        return None

    last_id = None
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# id:"):
                last_id = line.strip()

    return last_id


In [ ]:
def process_file(resume=True):
    examples = parse_conll(INPUT_PATH)
    total = len(examples)

    last_id = get_last_processed_id(OUTPUT_PATH) if resume else None
    skip = True if last_id else False

    for i, example in enumerate(examples, 1):
        ex_id, content = split_id_and_content(example)

        # skip logic
        if skip:
            if ex_id == last_id:
                skip = False
            continue

        try:
            translated = deepseek_chat(
                SYSTEM_PROMPT,
                content
            )

            with open(OUTPUT_PATH, "a", encoding="utf-8") as out:
                if ex_id:
                    out.write(ex_id + "\n")
                out.write(translated.strip() + "\n\n")

            if i % 50 == 0:
                print(f"✅ {i}/{total} done")

            time.sleep(0.7)

        except Exception as e:
            with open(LOG_PATH, "a", encoding="utf-8") as log:
                log.write(f"{ex_id} error: {e}\n")

On the first run

In [ ]:
process_file(resume=False)

def process_file(resume=True):
    examples = parse_conll(INPUT_PATH)
    total = len(examples)

    last_id = get_last_processed_id(OUTPUT_PATH) if resume else None
    skip = True if last_id else False

    for i, example in enumerate(examples, 1):
        ex_id, content = split_id_and_content(example)

        # skip logic
        if skip:
            if ex_id == last_id:
                skip = False
            continue

        try:
            translated = deepseek_chat(
                SYSTEM_PROMPT,
                content
            )

            with open(OUTPUT_PATH, "a", encoding="utf-8") as out:
                if ex_id:
                    out.write(ex_id + "\n")
                out.write(translated.strip() + "\n\n")

            if i % 50 == 0:
                print(f"{i}/{total} done")

            time.sleep(0.7)

        except Exception as e:
            with open(LOG_PATH, "a", encoding="utf-8") as log:
                log.write(f"{ex_id} error: {e}\n")

In [ ]:
process_file(resume=True)

On the first run

In [ ]:
with open("<your-base-dir>/input/train.conll", "r", encoding="utf-8") as fi:
    list_id = []
    for line in fi:
        if line.startswith("# id:"):
            last_id = line.strip()
            list_id.append(last_id)

with open("<your-base-dir>/output/train_tat.conll", "r", encoding="utf-8") as f:
    list_id_tat = []
    for line in f:
        if line.startswith("# id:"):
            last_id_tat = line.strip()
            list_id_tat.append(last_id_tat)

In [ ]:
train_path = "<your-base-dir>/input/train.conll"
cleaned_path = "<your-base-dir>/output/train_tat.conll"

# 1⃣ collect IDs from train
list_id = []
with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()
            list_id.append(clean_id)

# 2⃣ collect IDs from cleaned
list_id_tat = []
with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()
            list_id_tat.append(clean_id)

# 3⃣ find missing_ids
missing_ids = set(list_id) - set(list_id_tat)

print("Keeping only:", missing_ids)

# 4⃣ read train in blocks
with open(train_path, "r", encoding="utf-8") as f:
    sentences = f.read().strip().split("\n\n")

# 5⃣ keep ONLY missing_ids
filtered_sentences = []

for sent in sentences:
    for line in sent.split("\n"):
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()

            # NOTE: the condition is now reversed
            if clean_id in missing_ids:
                filtered_sentences.append(sent)
            break

print("Blocks before:", len(sentences))
print("Blocks after:", len(filtered_sentences))

# 6⃣ rewrite the file
with open(train_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(filtered_sentences))

print("Done")